In [ ]:
# ============================================================================
# STAGE 7a -- SAMPLE SUBGRAPHS -> REASONING CHAINS -> CSV   (NO LLM, NO GPU)
#
# STANDALONE single cell. Part 1 of the explainability eval (paper Sec. 4.2:
# "we sample N candidate reasoning subgraphs from the ensemble").
#
# For every question, across all 5 seed folders and both weighting schemes:
#   1. SAMPLE up to N_SUBGRAPHS proof subgraphs of the consensus answer by
#      RANDOM choice at every OR node (NOT top-k by weight). The paper says
#      "sample"; also, correlating a top-k-by-weight list against weights is
#      partly circular -- random sampling is the honest test.
#   2. DEDUPLICATE: two subgraphs whose node sets overlap by more than
#      DEDUP_JACCARD are treated as the same; only one is kept. This is the
#      fix for the near-identical-subgraph problem (e.g. gpqa_0011, whose
#      top-k subgraphs were all the same 27 nodes). A question keeps as many
#      DISTINCT subgraphs as sampling found, capped at N_SUBGRAPHS; questions
#      left with < MIN_SUBGRAPHS distinct ones are dropped.
#   3. LINEARIZE each subgraph into a readable REASONING CHAIN -- its nodes in
#      topological order (Facts/Planning first, Conclusion/Answer last), one
#      numbered step per node. The full node text is kept verbatim.
#   4. Record the ENSEMBLE-WEIGHT score Phi(P) = sum_v W(v) for each subgraph.
#
# Writes TWO CSVs (no model loaded). Every text field is written with
# QUOTE_ALL so embedded newlines / commas round-trip intact.
#   <OUT_DIR>/stage7_chains_per_subgraph.csv
#       one row per sampled subgraph -- the full chain text + weight score/rank.
#   <OUT_DIR>/stage7_chains_per_question.csv
#       one row per question -- all its chains inline + the weight ranking.
#
# Stage 7b reads the per-subgraph CSV, asks Qwen2.5-32B to rank each
# question's chains in ONE call, and correlates that with the weight ranking.
# ============================================================================

import csv, glob, json, os, random
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple, FrozenSet

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEED_DIRS = [
    "./outputs_1234",
    "./outputs_2234",
    "./outputs_3234",
    "./outputs_4234",
    "./outputs_5234",
]

SCHEME_TAGS      = ["simple", "accuracy"]
DATASETS_TO_EVAL = ["musr_ta"]      # ArgKP excluded: answer echoes stance

# Where the two CSVs are written. Defaults to the first seed dir.
OUT_DIR = SEED_DIRS[0]

N_SUBGRAPHS    = 5        # subgraphs sampled per question (randomly)
MIN_SUBGRAPHS  = 3        # need this many DISTINCT ones or the question is dropped
DEDUP_JACCARD  = 0.90     # node-set Jaccard > this  => "same" subgraph
SAMPLE_SEED    = 12345    # makes the random subgraph sampling reproducible
N_SAMPLE_DRAWS = 80       # random unfoldings drawn before dedup. Oversampled so
                          # that after dedup we still usually reach N_SUBGRAPHS
                          # (=5); questions with few OR-branches simply yield
                          # fewer distinct subgraphs and may fall back toward
                          # MIN_SUBGRAPHS.

PER_SUBGRAPH_CSV  = os.path.join(OUT_DIR, "stage7_chains_per_subgraph.csv")
PER_QUESTION_CSV  = os.path.join(OUT_DIR, "stage7_chains_per_question.csv")


# ============================================================================
# 1. RANDOM SUBGRAPH SAMPLING
#
# A proof subgraph is built by unfolding from the target (the Answer node):
#   - Atomic node            -> leaf, stop.
#   - And node (1 support)   -> include the unique support bundle, expand all.
#   - Or node (>1 support)   -> pick ONE support bundle AT RANDOM, expand all.
# Repeating with fresh randomness yields different subgraphs whenever the
# proof tree contains Or nodes. We draw N_SAMPLE_DRAWS unfoldings, then dedup.
# ============================================================================
def sample_one_subgraph(target: str,
                         id_to_node: Dict[str, Dict],
                         rng: random.Random) -> Dict:
    """One random proof subgraph rooted at `target`.
    Returns {"nodes": [ids], "edges": [[child,parent],...]}."""
    nodes: Set[str] = set()
    edges: List[Tuple[str, str]] = []

    def unfold(nid: str, on_path: FrozenSet[str]) -> None:
        if nid in on_path:          # cycle guard (Stage 3 broke cycles anyway)
            nodes.add(nid)
            return
        node = id_to_node.get(nid)
        if node is None:
            nodes.add(nid)
            return
        nodes.add(nid)
        gate = node.get("gate", "Atomic")
        support = node.get("support", [])
        if gate == "Atomic" or not support:
            return
        if gate == "And":
            chosen = support[0]
        else:                       # Or -> random support bundle
            chosen = support[rng.randrange(len(support))]
        new_path = on_path | {nid}
        for m in chosen.get("members", []):
            edges.append((nid, m))
            unfold(m, new_path)

    unfold(target, frozenset())
    return {"nodes": sorted(nodes), "edges": [[c, p] for (c, p) in edges]}


def sample_distinct_subgraphs(target: str,
                              id_to_node: Dict[str, Dict],
                              W: Dict[str, float],
                              k: int,
                              n_draws: int,
                              dedup_jaccard: float,
                              rng: random.Random) -> List[Dict]:
    """Draw n_draws random subgraphs, dedup by node-set Jaccard, return up to
    k distinct ones. Each result carries node_details + the weight score."""
    kept: List[Dict] = []
    kept_sets: List[FrozenSet[str]] = []

    for _ in range(n_draws):
        g = sample_one_subgraph(target, id_to_node, rng)
        nodeset = frozenset(g["nodes"])
        # dedup: skip if too similar to something already kept
        is_dup = False
        for ks in kept_sets:
            inter = len(nodeset & ks)
            union = len(nodeset | ks)
            if union > 0 and inter / union > dedup_jaccard:
                is_dup = True
                break
        if is_dup:
            continue
        g["node_details"] = [{
            "id":   nid,
            "type": id_to_node.get(nid, {}).get("type"),
            "text": id_to_node.get(nid, {}).get("text"),
            "W":    round(float(id_to_node.get(nid, {}).get("W", 0.0)), 6),
            "gate": id_to_node.get(nid, {}).get("gate"),
        } for nid in g["nodes"]]
        g["score"] = round(sum(W.get(x, 0.0) for x in g["nodes"]), 6)
        kept.append(g)
        kept_sets.append(nodeset)
        if len(kept) >= k:
            break
    return kept


# ============================================================================
# 2. CONSENSUS CONCLUSION + ENSEMBLE LOOKUP
# ============================================================================
def consensus_conclusion(parsed_rec: Dict) -> Optional[Dict]:
    consensus = parsed_rec.get("consensus_answer")
    for pc in parsed_rec.get("per_conclusion", []) or []:
        if pc.get("answer_text") == consensus:
            return pc
    pcs = parsed_rec.get("per_conclusion", []) or []
    return pcs[0] if pcs else None


def load_weighted_index(seed_dir: str, dataset: str, scheme: str
                        ) -> Dict[str, Dict]:
    """question_id -> Stage 4 weighted record (carries the weighted DAG)."""
    path = os.path.join(seed_dir,
                        f"{dataset}_ensemble_dags_weighted_{scheme}.jsonl")
    idx: Dict[str, Dict] = {}
    if not os.path.exists(path):
        return idx
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            qid = r.get("question_id")
            if qid is not None:
                idx[qid] = r
    return idx


# ============================================================================
# 3. LINEARIZE A SUBGRAPH INTO A REASONING CHAIN
#
# Topological order: a node appears only after every node that supports it,
# so Facts/Planning come first and the Conclusion/Answer comes last. Each
# node becomes one numbered step "[Type] full text". Nothing is truncated.
#
# Returns (chain_text, ok) where ok is True only if EVERY node in the
# subgraph appears exactly once in the chain AND every node's full text is
# present verbatim. The caller asserts ok -- so a silent truncation anywhere
# becomes a hard, visible failure instead of corrupting the eval.
# ============================================================================
def linearize_chain(graph: Dict) -> Tuple[str, bool]:
    details = {d["id"]: d for d in (graph.get("node_details") or [])}
    node_ids = list(graph.get("nodes", []))
    idset = set(node_ids)

    # edges are [child, parent] meaning parent supports child.
    preds: Dict[str, Set[str]] = {nid: set() for nid in node_ids}
    for edge in graph.get("edges", []):
        if len(edge) != 2:
            continue
        child, parent = edge
        if child in idset and parent in idset:
            preds[child].add(parent)

    # Kahn topological sort: parents (support) before children.
    indeg = {nid: len(preds[nid]) for nid in node_ids}
    children: Dict[str, List[str]] = defaultdict(list)
    for child, ps in preds.items():
        for p in ps:
            children[p].append(child)
    queue = [nid for nid in node_ids if indeg[nid] == 0]
    order: List[str] = []
    qi = 0
    while qi < len(queue):
        u = queue[qi]; qi += 1
        order.append(u)
        for c in children[u]:
            indeg[c] -= 1
            if indeg[c] == 0:
                queue.append(c)
    # any leftover (shouldn't happen -- Stage 3 broke cycles) appended stably
    if len(order) < len(node_ids):
        seen = set(order)
        order += [nid for nid in node_ids if nid not in seen]

    lines: List[str] = []
    for i, nid in enumerate(order, start=1):
        d = details.get(nid, {})
        ntype = d.get("type") or "?"
        ntext = (d.get("text") or "").strip()
        lines.append(f"Step {i} [{ntype}]: {ntext}")
    chain = "\n".join(lines)

    # ---- no-truncation verification ---------------------------------------
    # 1. every node appears exactly once (order is a permutation of node_ids)
    ok = (sorted(order) == sorted(node_ids)) and (len(order) == len(set(order)))
    # 2. every node's full text is present verbatim in the chain
    if ok:
        for nid in node_ids:
            full = (details.get(nid, {}).get("text") or "").strip()
            if full and full not in chain:
                ok = False
                break
    return chain, ok


# ============================================================================
# 4. RANKING HELPERS
# ============================================================================
def to_rank(scores: List[float]) -> List[int]:
    """1 = best (highest score). Stable tie-break by index."""
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    rk = [0] * len(scores)
    for pos, i in enumerate(order):
        rk[i] = pos + 1
    return rk


# ============================================================================
# DRIVER
# ============================================================================
PER_SUBGRAPH_FIELDS = [
    "row_uid",            # globally unique: seed|scheme|dataset|qid|subgraph_idx
    "seed_idx", "seed_dir", "scheme", "dataset", "question_id",
    "consensus_answer", "gold_label", "answer_correct",
    "subgraph_idx",       # 0..N-1 within the question
    "n_nodes",
    "weight_score",       # Phi(P) = sum_v W(v)
    "weight_rank",        # 1 = highest weight among this question's subgraphs
    "n_subgraphs_for_question",
    "reasoning_chain",    # the full linearized chain -- never truncated
]

PER_QUESTION_FIELDS = [
    "question_uid",       # seed|scheme|dataset|qid
    "seed_idx", "scheme", "dataset", "question_id",
    "consensus_answer", "gold_label", "answer_correct",
    "n_subgraphs",
    "weight_scores",      # json list, aligned to subgraph_idx 0..N-1
    "weight_ranking",     # json list of ranks, aligned to subgraph_idx
    "chains_json",        # json list of full chain strings, aligned to idx
]

print("[stage 7a] sampling random subgraphs and building reasoning chains ...")
rng = random.Random(SAMPLE_SEED)

per_subgraph_rows: List[Dict] = []
per_question_rows: List[Dict] = []

n_q_total = 0
n_q_kept = 0
n_q_dropped_fewsub = 0
n_q_dropped_noconc = 0
n_chain_verify_fail = 0   # questions skipped because a chain lost node text

for seed_idx, seed_dir in enumerate(SEED_DIRS):
    if not os.path.isdir(seed_dir):
        print(f"  [seed {seed_idx}] MISSING dir, skipping: {seed_dir}")
        continue
    for scheme in SCHEME_TAGS:
        for dataset in DATASETS_TO_EVAL:
            parsed_path = os.path.join(seed_dir,
                                       f"{dataset}_parsed_{scheme}.jsonl")
            if not os.path.exists(parsed_path):
                print(f"  [seed {seed_idx}/{scheme}/{dataset}] no parsed file, "
                      f"skipping")
                continue

            weighted_idx = load_weighted_index(seed_dir, dataset, scheme)

            n_q = n_kept = n_few = n_noc = 0
            with open(parsed_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        prec = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    n_q += 1
                    n_q_total += 1

                    qid = prec.get("question_id")
                    pc = consensus_conclusion(prec)
                    wrec = weighted_idx.get(qid)
                    if pc is None or wrec is None:
                        n_noc += 1
                        n_q_dropped_noconc += 1
                        continue

                    ensemble = wrec.get("ensemble_dag") or {}
                    nodes = ensemble.get("nodes", [])
                    id_to_node = {n["id"]: n for n in nodes}
                    W = {n["id"]: float(n.get("W", 0.0)) for n in nodes}
                    target = pc.get("answer_cluster_id")
                    if target is None or target not in id_to_node:
                        n_noc += 1
                        n_q_dropped_noconc += 1
                        continue

                    subs = sample_distinct_subgraphs(
                        target, id_to_node, W,
                        k=N_SUBGRAPHS, n_draws=N_SAMPLE_DRAWS,
                        dedup_jaccard=DEDUP_JACCARD, rng=rng)
                    if len(subs) < MIN_SUBGRAPHS:
                        n_few += 1
                        n_q_dropped_fewsub += 1
                        continue

                    weight_scores = [g["score"] for g in subs]
                    weight_ranking = to_rank(weight_scores)
                    chains, chain_oks = [], []
                    for g in subs:
                        ch, ok = linearize_chain(g)
                        chains.append(ch)
                        chain_oks.append(ok)
                    if not all(chain_oks):
                        n_chain_verify_fail += 1
                        # a chain lost node text -- do NOT write a corrupt
                        # question; skip it and record so it's visible.
                        continue

                    gold = (prec.get("gold_label") or "").strip().lower()
                    cons = (prec.get("consensus_answer") or "").strip().lower()
                    answer_correct = bool(gold) and bool(cons) and gold == cons

                    quid = f"{seed_idx}|{scheme}|{dataset}|{qid}"
                    for si, g in enumerate(subs):
                        per_subgraph_rows.append({
                            "row_uid":  f"{quid}|{si}",
                            "seed_idx": seed_idx, "seed_dir": seed_dir,
                            "scheme":   scheme, "dataset": dataset,
                            "question_id": qid,
                            "consensus_answer": prec.get("consensus_answer"),
                            "gold_label": prec.get("gold_label"),
                            "answer_correct": answer_correct,
                            "subgraph_idx": si,
                            "n_nodes": len(g["nodes"]),
                            "weight_score": weight_scores[si],
                            "weight_rank":  weight_ranking[si],
                            "n_subgraphs_for_question": len(subs),
                            "reasoning_chain": chains[si],
                        })
                    per_question_rows.append({
                        "question_uid": quid,
                        "seed_idx": seed_idx, "scheme": scheme,
                        "dataset": dataset, "question_id": qid,
                        "consensus_answer": prec.get("consensus_answer"),
                        "gold_label": prec.get("gold_label"),
                        "answer_correct": answer_correct,
                        "n_subgraphs": len(subs),
                        "weight_scores": json.dumps(weight_scores),
                        "weight_ranking": json.dumps(weight_ranking),
                        "chains_json": json.dumps(chains, ensure_ascii=False),
                    })
                    n_kept += 1
                    n_q_kept += 1

            print(f"  [seed {seed_idx}/{scheme}/{dataset}] "
                  f"{n_q} questions: {n_kept} kept, "
                  f"{n_few} dropped (<{MIN_SUBGRAPHS} distinct subgraphs), "
                  f"{n_noc} dropped (no consensus/DAG)")

# ----------------------------------------------------------------------------
# Write the CSVs. QUOTE_ALL + newline='' so multi-line chain text survives
# round-trip intact. No field is truncated.
# ----------------------------------------------------------------------------
print(f"\n[stage 7a] writing per-subgraph CSV: {PER_SUBGRAPH_CSV}")
with open(PER_SUBGRAPH_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=PER_SUBGRAPH_FIELDS,
                       quoting=csv.QUOTE_ALL)
    w.writeheader()
    for row in per_subgraph_rows:
        w.writerow(row)

print(f"[stage 7a] writing per-question CSV: {PER_QUESTION_CSV}")
with open(PER_QUESTION_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=PER_QUESTION_FIELDS,
                       quoting=csv.QUOTE_ALL)
    w.writeheader()
    for row in per_question_rows:
        w.writerow(row)

# ----------------------------------------------------------------------------
# NO-TRUNCATION CSV ROUND-TRIP CHECK.
# Read the per-subgraph CSV straight back in and confirm every reasoning_chain
# is byte-identical to what we held in memory. If the csv module's field-size
# limit or any quoting issue had clipped a field, this catches it. We raise
# the field-size limit first because long multi-line chains exceed the
# default 131072-char cap.
# ----------------------------------------------------------------------------
import sys as _sys
csv.field_size_limit(_sys.maxsize)

print(f"\n[stage 7a] verifying CSV round-trip (no chain truncated) ...")
in_memory = {r["row_uid"]: r["reasoning_chain"] for r in per_subgraph_rows}
mismatches = 0
checked = 0
with open(PER_SUBGRAPH_CSV, "r", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        checked += 1
        uid = row["row_uid"]
        got = row["reasoning_chain"]
        expect = in_memory.get(uid)
        if expect is None or got != expect:
            mismatches += 1
            if mismatches <= 5:
                el = len(expect) if expect is not None else -1
                print(f"  MISMATCH row_uid={uid}: "
                      f"in-memory {el} chars, read-back {len(got)} chars")
if mismatches == 0:
    print(f"  OK -- all {checked} chains survived the CSV round-trip "
          f"byte-identical. Nothing was truncated.")
else:
    raise RuntimeError(
        f"{mismatches}/{checked} chains differ after CSV round-trip -- "
        f"truncation/quoting bug. Do NOT use these CSVs.")

# ----------------------------------------------------------------------------
# Summary.
# ----------------------------------------------------------------------------
print(f"\n[stage 7a] done.")
print(f"  questions seen          : {n_q_total}")
print(f"  questions kept          : {n_q_kept}")
print(f"  dropped (<{MIN_SUBGRAPHS} subgraphs)  : {n_q_dropped_fewsub}")
print(f"  dropped (no consensus)  : {n_q_dropped_noconc}")
print(f"  dropped (chain verify)  : {n_chain_verify_fail}")
print(f"  per-subgraph rows       : {len(per_subgraph_rows)}")
print(f"  per-question rows       : {len(per_question_rows)}")
if n_chain_verify_fail:
    print(f"  NOTE: {n_chain_verify_fail} question(s) were skipped because a "
          f"linearized chain did not contain every node's full text. This "
          f"should be 0; investigate if not.")

# chain-length sanity check (so you can size Stage 7b's context window)
if per_subgraph_rows:
    lens = [len(r["reasoning_chain"]) for r in per_subgraph_rows]
    lens.sort()
    n = len(lens)
    def pct(p): return lens[min(n - 1, int(p * n))]
    print(f"\n[stage 7a] reasoning-chain length (characters):")
    print(f"  min/median/p95/p99/max : "
          f"{lens[0]} / {lens[n//2]} / {pct(0.95)} / {pct(0.99)} / {lens[-1]}")
    print(f"  (Stage 7b uses these to size the judge context so NO chain is "
          f"ever truncated when sent to the LLM.)")
print(f"\n[stage 7a] CSVs ready for Stage 7b:")
print(f"  {PER_SUBGRAPH_CSV}")
print(f"  {PER_QUESTION_CSV}")

In [ ]:
# ============================================================================
# STAGE 7b -- LLM RANKS THE REASONING CHAINS, CORRELATE WITH WEIGHT RANKING
#
# STANDALONE single cell. Part 2 of the explainability eval (paper Sec. 4.2).
# Reads Stage 7a's per-subgraph CSV. For every question:
#   1. Collect its N reasoning chains (full text -- Stage 7a never truncated).
#   2. Ask Qwen3-32B to RANK all N chains in ONE call, by reasoning and
#      justification quality. The chains are presented in a RANDOM order
#      (de-biasing) with the weight scores HIDDEN, so the judge cannot see the
#      ranking it is being validated against.
#   3. Parse the judge's ranking, map it back to the original subgraph order.
#   4. Correlate the judge ranking with the ensemble-weight ranking from
#      Stage 7a (Spearman rho + Kendall tau), per question.
#
# JUDGE = Qwen3-32B. Qwen3 has THINKING MODE ON BY DEFAULT and emits
# <think>...</think> reasoning tokens. We disable thinking via the chat
# template (enable_thinking=False) AND strip any <think> block before parsing,
# so reasoning tokens can never corrupt the parsed ranking.
#
# NO CHAIN TEXT IS EVER TRUNCATED WHEN SENT TO THE LLM. Instead:
#   - JUDGE_MAX_LEN is set large (32k) so realistic prompts fit with headroom.
#   - Every prompt is token-measured BEFORE dispatch. If a prompt still would
#     not fit, that QUESTION IS DROPPED from the eval and logged -- we never
#     feed the model a clipped chain. A dropped question is visible and
#     honest; a truncated chain silently corrupts the ranking.
#
# Aggregation: per-seed mean +/- std AND a pooled correlation, per
# (scheme, dataset). Writes a per-question results CSV + a summary JSON.
#
# Set HF_*/VLLM_* env vars in the shell BEFORE launching Jupyter (see Stage 2).
# ============================================================================

# ----------------------------------------------------------------------------
# CACHE REDIRECTION + SANITY CHECK
# ----------------------------------------------------------------------------
import os

os.environ["VLLM_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_MOE_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_DEEP_GEMM_WARMUP"] = "skip"

HF_CACHE_ROOT = "/work/hdd/bfrc"
os.environ["HF_HOME"]                 = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_HUB_CACHE"]            = f"{HF_CACHE_ROOT}/hf/hub"
os.environ["TRANSFORMERS_CACHE"]      = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_DATASETS_CACHE"]       = f"{HF_CACHE_ROOT}/hf/datasets"
os.environ["VLLM_CACHE_ROOT"]         = f"{HF_CACHE_ROOT}/vllm"
os.environ["TRITON_CACHE_DIR"]        = f"{HF_CACHE_ROOT}/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{HF_CACHE_ROOT}/torch_inductor"
os.environ["TMPDIR"]                  = f"{HF_CACHE_ROOT}/tmp"
for p in (os.environ["HF_HOME"], os.environ["HF_HUB_CACHE"],
          os.environ["HF_DATASETS_CACHE"], os.environ["VLLM_CACHE_ROOT"],
          os.environ["TRITON_CACHE_DIR"], os.environ["TORCHINDUCTOR_CACHE_DIR"],
          os.environ["TMPDIR"]):
    os.makedirs(p, exist_ok=True)

import importlib, sys
if "huggingface_hub" in sys.modules:
    importlib.reload(sys.modules["huggingface_hub"])
    if "huggingface_hub.constants" in sys.modules:
        importlib.reload(sys.modules["huggingface_hub.constants"])
from huggingface_hub import constants as _hf_constants
_resolved = str(_hf_constants.HF_HUB_CACHE)
print(f"[cache check] huggingface_hub resolved HF_HUB_CACHE = {_resolved}")
if not _resolved.startswith(HF_CACHE_ROOT):
    print(f"[cache check] WARNING: HF cache outside {HF_CACHE_ROOT}. Restart "
          f"the kernel with env vars set BEFORE launching Jupyter.")
else:
    print(f"[cache check] OK -- writing under {HF_CACHE_ROOT}")


# ----------------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------------
import csv, gc, json, re, time, random
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

try:
    from scipy.stats import spearmanr, kendalltau
    _HAVE_SCIPY = True
except ImportError:
    _HAVE_SCIPY = False
    print("[stage 7b] scipy not found -- using built-in rank-correlation.")

# csv module's default field-size limit (131072) is too small for long
# multi-line chain fields. Raise it so reading the CSV never throws.
csv.field_size_limit(sys.maxsize)


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
# Stage 7a wrote these. Point at the same OUT_DIR Stage 7a used.
STAGE7A_DIR      = "./outputs_1234"
PER_SUBGRAPH_CSV = os.path.join(STAGE7A_DIR, "stage7_chains_per_subgraph.csv")

# Outputs.
RESULTS_CSV   = os.path.join(STAGE7A_DIR, "stage7_ranking_results.csv")
SUMMARY_JSON  = os.path.join(STAGE7A_DIR, "stage7_ranking_summary.json")

# A question is "right-answer-wrong-reason" if its consensus answer is correct
# but the judge<->weight rank correlation is at/below this threshold. Set this
# to a genuine inversion cutoff (NOT 0.0 -- with a true corr near zero, a 0.0
# threshold just splits noise in half and the count is meaningless).
RAWR_RHO_THRESHOLD = -0.30

# Order de-biasing: present each question's chains to the judge in a random
# order, this many independent times, and average the resulting rankings.
# >1 cancels the judge's positional bias. 1 = single pass (faster).
N_ORDER_REPEATS = 5
ORDER_SEED      = 999

# Judge model. Qwen3-32B: thinking mode is ON by default and emits
# <think>...</think> tokens. We disable it via the chat template
# (ENABLE_THINKING=False) AND strip any <think> block before parsing.
JUDGE_MODEL    = "Qwen/Qwen3-32B"
ENABLE_THINKING = False
JUDGE_TEMP     = 0.0
JUDGE_TOP_P    = 1.0
JUDGE_MAX_TOK  = 256      # raised: a reasoning model may emit a short
                          # preamble before the ranking. The parser tolerates
                          # surrounding text; the ranking itself is tiny.
JUDGE_MAX_LEN  = 32768    # large on purpose: full chains, never truncated
JUDGE_GPU_UTIL = 0.90
JUDGE_TP       = 1        # bump for multi-GPU

VLLM_DOWNLOAD_DIR = f"{HF_CACHE_ROOT}/hf/hub"
os.makedirs(VLLM_DOWNLOAD_DIR, exist_ok=True)


# ============================================================================
# 1. JUDGE PROMPT  (one-shot N-way ranking)
# ============================================================================
JUDGE_SYSTEM = (
    "You are a strict evaluator of reasoning quality. You are given several "
    "candidate reasoning chains that all argue toward the SAME final answer. "
    "Rank them by the QUALITY of their reasoning and justification: are the "
    "steps well-grounded in facts, are the inferences logically sound, are "
    "there gaps, leaps, or irrelevant steps. Do NOT reward a chain for "
    "reaching the answer -- they all reach the same answer. A chain can reach "
    "a correct answer through weak or wrong reasoning; rank that chain low."
    "The number of steps does NOT matter; do not prefer a chain merely because it is longer."
)


def build_ranking_prompt(question_blob: str, final_answer: str,
                          ordered_chains: List[str]) -> str:
    """ordered_chains are presented as CHAIN 1..N in the given (random) order.
    The judge ranks by these presentation labels."""
    blocks = []
    for i, ch in enumerate(ordered_chains, start=1):
        blocks.append(f"--- CHAIN {i} ---\n{ch}")
    chains_block = "\n\n".join(blocks)
    n = len(ordered_chains)
    return f"""You are given {n} candidate reasoning chains. They ALL conclude the
same final answer. Rank them from BEST to WORST reasoning quality.

QUESTION CONTEXT:
{question_blob}

FINAL ANSWER (every chain reaches this -- it is NOT what you are judging):
{final_answer}

{chains_block}

Rank all {n} chains from best reasoning to worst. Output ONLY the ranking as
chain numbers separated by '>', best first. For example: 2 > 1 > 3
Do not output anything else."""


# ============================================================================
# 2. PARSE THE JUDGE'S RANKING  (Qwen3-aware: strips <think> reasoning)
# ============================================================================
def _strip_reasoning(raw: str) -> str:
    """Remove any <think>...</think> block (closed or dangling) so digits
    inside a Qwen3 reasoning block never reach the ranking parser."""
    s = re.sub(r"<think>.*?</think>", " ", raw, flags=re.DOTALL | re.IGNORECASE)
    if "</think>" in s.lower():
        s = s[s.lower().rfind("</think>") + len("</think>"):]
    if "<think>" in s.lower():
        s = s[:s.lower().find("<think>")]
    return s.strip()


def parse_ranking(raw: str, n: int) -> Optional[List[int]]:
    """Parse 'best > ... > worst' into a 0-indexed presentation-order list,
    best first. Returns None if the response is not a valid permutation of
    1..n. Strips any reasoning block, then prefers the LAST line containing
    '>' so stray preamble digits do not corrupt the parse."""
    cleaned = _strip_reasoning(raw)
    if not cleaned:
        return None
    candidate_text = cleaned
    rank_lines = [ln for ln in cleaned.splitlines() if ">" in ln]
    if rank_lines:
        candidate_text = rank_lines[-1]
    nums = [int(x) for x in re.findall(r"\d+", candidate_text)]
    # keep only the first clean run that is a permutation of 1..n
    seen, perm = set(), []
    for x in nums:
        if 1 <= x <= n and x not in seen:
            seen.add(x)
            perm.append(x)
        if len(perm) == n:
            break
    if len(perm) != n or set(perm) != set(range(1, n + 1)):
        return None
    return [x - 1 for x in perm]   # 0-indexed presentation slots, best first


def apply_template(tok, messages) -> str:
    """Render the chat template, disabling Qwen3 thinking when supported.
    Falls back gracefully for tokenizers without the enable_thinking kwarg
    (the <think> stripping in parse_ranking is the safety net either way)."""
    try:
        return tok.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
            enable_thinking=ENABLE_THINKING)
    except TypeError:
        return tok.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False)


# ============================================================================
# 3. RANK CORRELATION
# ============================================================================
def _rankdata(a: List[float]) -> List[float]:
    order = sorted(range(len(a)), key=lambda i: a[i])
    ranks = [0.0] * len(a)
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and a[order[j + 1]] == a[order[i]]:
            j += 1
        avg = (i + j) / 2.0 + 1.0
        for kk in range(i, j + 1):
            ranks[order[kk]] = avg
        i = j + 1
    return ranks


def rank_correlations(x: List[float], y: List[float]) -> Tuple[float, float]:
    n = len(x)
    if n < 2:
        return float("nan"), float("nan")
    if _HAVE_SCIPY:
        rho, _ = spearmanr(x, y)
        tau, _ = kendalltau(x, y)
        return (float(rho) if rho == rho else float("nan"),
                float(tau) if tau == tau else float("nan"))
    rx, ry = _rankdata(x), _rankdata(y)
    mx, my = sum(rx) / n, sum(ry) / n
    sxy = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    sxx = sum((a - mx) ** 2 for a in rx)
    syy = sum((b - my) ** 2 for b in ry)
    rho = sxy / (sxx ** 0.5 * syy ** 0.5) if sxx > 0 and syy > 0 else float("nan")
    conc = disc = 0
    for i in range(n):
        for j in range(i + 1, n):
            s = (x[i] - x[j]) * (y[i] - y[j])
            if s > 0:
                conc += 1
            elif s < 0:
                disc += 1
    tau = (conc - disc) / (0.5 * n * (n - 1)) if n > 1 else float("nan")
    return rho, tau


def mean_std(vals: List[float]) -> Tuple[float, float]:
    v = [x for x in vals if x == x]
    if not v:
        return float("nan"), float("nan")
    return float(np.mean(v)), float(np.std(v))


# ============================================================================
# 4. LOAD STAGE 7a's PER-SUBGRAPH CSV  ->  per-question groups
# ============================================================================
print(f"\n[stage 7b] reading {PER_SUBGRAPH_CSV}")
if not os.path.exists(PER_SUBGRAPH_CSV):
    raise RuntimeError(f"Stage 7a CSV not found: {PER_SUBGRAPH_CSV}. "
                       f"Run Stage 7a first.")

# question_uid -> ordered list of subgraph rows (subgraph_idx 0..N-1)
questions: Dict[str, List[Dict]] = defaultdict(list)
with open(PER_SUBGRAPH_CSV, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        quid = "|".join([row["seed_idx"], row["scheme"],
                         row["dataset"], row["question_id"]])
        questions[quid].append(row)

# sort each question's rows by subgraph_idx so indices are stable
for quid in questions:
    questions[quid].sort(key=lambda r: int(r["subgraph_idx"]))

print(f"[stage 7b] {len(questions)} questions loaded")


# ============================================================================
# 5. LOAD TOKENIZER, BUILD PROMPTS, DROP (NEVER TRUNCATE) OVERLONG QUESTIONS
# ============================================================================
print(f"\n[stage 7b] loading tokenizer {JUDGE_MODEL} (for length checking)")
tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL, cache_dir=VLLM_DOWNLOAD_DIR, trust_remote_code=True)

# verify the thinking switch had an effect (informational only)
try:
    _probe_on = tokenizer.apply_chat_template(
        [{"role": "user", "content": "x"}], add_generation_prompt=True,
        tokenize=False, enable_thinking=True)
    _probe_off = tokenizer.apply_chat_template(
        [{"role": "user", "content": "x"}], add_generation_prompt=True,
        tokenize=False, enable_thinking=False)
    if _probe_on != _probe_off:
        print(f"[stage 7b] thinking switch supported; using "
              f"enable_thinking={ENABLE_THINKING}")
    else:
        print(f"[stage 7b] note: enable_thinking accepted but template "
              f"unchanged -- relying on <think> stripping in the parser.")
except TypeError:
    print(f"[stage 7b] note: tokenizer has no enable_thinking kwarg -- "
          f"relying entirely on <think> stripping in the parser.")

rng = random.Random(ORDER_SEED)
prompt_budget = JUDGE_MAX_LEN - JUDGE_MAX_TOK - 32   # leave room for generation

# One judge call per (question, order-repeat). Each carries the random
# presentation permutation so we can map the judge's answer back.
flat_prompts: List[str] = []
flat_meta: List[Dict] = []        # {quid, perm}  perm: presentation slot -> orig idx

n_dropped_toolong = 0
n_chain_missing = 0          # prompts where a chain was not present verbatim
dropped_examples: List[str] = []

for quid, rows in questions.items():
    n = len(rows)
    chains_orig = [r["reasoning_chain"] for r in rows]
    qblob = (f"[dataset={rows[0]['dataset']}, "
             f"question_id={rows[0]['question_id']}]")
    final_answer = rows[0].get("consensus_answer", "")

    fits = True
    chain_ok = True
    built = []   # (prompt_str, perm)
    for _ in range(N_ORDER_REPEATS):
        perm = list(range(n))
        rng.shuffle(perm)                       # presentation slot -> orig idx
        ordered_chains = [chains_orig[orig] for orig in perm]
        user = build_ranking_prompt(qblob, final_answer, ordered_chains)
        messages = [{"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user",   "content": user}]
        rendered = apply_template(tokenizer, messages)

        # NO-TRUNCATION GUARD 1: every full chain must appear verbatim in the
        # prompt actually being sent. Catches any accidental clipping in
        # prompt assembly or chat templating.
        for ch in ordered_chains:
            if ch.strip() and ch not in rendered:
                n_chain_missing += 1
                chain_ok = False
                break
        if not chain_ok:
            break

        # NO-TRUNCATION GUARD 2: the prompt must fit the context window. We
        # never clip a chain to make it fit -- we drop the whole question.
        ntok = len(tokenizer.encode(rendered, add_special_tokens=False))
        if ntok > prompt_budget:
            fits = False
            break
        built.append((rendered, perm))

    if not chain_ok:
        continue                       # hard-abort happens after the loop
    if not fits:
        n_dropped_toolong += 1
        if len(dropped_examples) < 10:
            dropped_examples.append(quid)
        continue

    for (rendered, perm) in built:
        flat_prompts.append(rendered)
        flat_meta.append({"quid": quid, "perm": perm})

print(f"[stage 7b] {len(flat_prompts)} judge prompts queued "
      f"({N_ORDER_REPEATS} order-repeat(s) per question)")
if n_dropped_toolong:
    print(f"[stage 7b] DROPPED {n_dropped_toolong} questions whose prompt "
          f"exceeded {prompt_budget} tokens -- NOT truncated, excluded so the "
          f"judge never sees a clipped chain.")
    print(f"           examples: {dropped_examples}")
    print(f"           (if this is many, raise JUDGE_MAX_LEN.)")
if n_chain_missing:
    raise RuntimeError(
        f"{n_chain_missing} prompt(s) did not contain a chain verbatim -- "
        f"a chain was altered/clipped during prompt assembly. Aborting so no "
        f"truncated chain reaches the judge. Investigate before re-running.")
else:
    print(f"[stage 7b] no-truncation check: every chain appears verbatim in "
          f"its prompt.")


# ============================================================================
# 6. RUN THE JUDGE  (one batched vLLM call)
# ============================================================================
if not flat_prompts:
    print("[stage 7b] nothing to rank -- check the CSV / config.")
else:
    print(f"\n[stage 7b] loading judge {JUDGE_MODEL}")
    llm = LLM(
        model=JUDGE_MODEL,
        dtype="bfloat16",
        trust_remote_code=True,
        gpu_memory_utilization=JUDGE_GPU_UTIL,
        max_model_len=JUDGE_MAX_LEN,
        tensor_parallel_size=JUDGE_TP,
        download_dir=VLLM_DOWNLOAD_DIR,
    )
    sps = [SamplingParams(temperature=JUDGE_TEMP, top_p=JUDGE_TOP_P,
                          max_tokens=JUDGE_MAX_TOK, seed=0, n=1)
           for _ in flat_prompts]

    print(f"[stage 7b] dispatching {len(flat_prompts)} ranking prompts ...")
    t0 = time.time()
    outputs = llm.generate(flat_prompts, sps)
    dt = time.time() - t0
    print(f"[stage 7b] judge done in {dt:.1f}s "
          f"({dt/max(1,len(flat_prompts)):.3f}s/call avg)")

    # ---- map each judge ranking back to original subgraph indices ----------
    # For a question we average a "rank value" per original subgraph across
    # all order-repeats: judged_rank[orig] in 1..N, 1 = best. Lower = better.
    judged_rank_sum: Dict[str, List[float]] = {}
    judged_rank_cnt: Dict[str, int] = defaultdict(int)
    n_unparsed = 0

    for meta, out in zip(flat_meta, outputs):
        quid = meta["quid"]
        perm = meta["perm"]                 # presentation slot -> orig idx
        n = len(perm)
        parsed = parse_ranking(out.outputs[0].text, n)
        if quid not in judged_rank_sum:
            judged_rank_sum[quid] = [0.0] * n
        if parsed is None:
            n_unparsed += 1
            continue
        # parsed = presentation slots, best first. position 0 = best.
        for position, pres_slot in enumerate(parsed):
            orig = perm[pres_slot]
            judged_rank_sum[quid][orig] += (position + 1)   # 1 = best
        judged_rank_cnt[quid] += 1

    if n_unparsed:
        pct = 100.0 * n_unparsed / max(1, len(flat_meta))
        print(f"[stage 7b] note: {n_unparsed}/{len(flat_meta)} judge outputs "
              f"({pct:.1f}%) were unparseable (not a valid permutation) -- "
              f"those repeats skipped")
        if pct > 10:
            print(f"[stage 7b] WARNING: high parse-failure rate. With a "
                  f"reasoning model, check that <think> stripping works and "
                  f"that JUDGE_MAX_TOK ({JUDGE_MAX_TOK}) is large enough for "
                  f"the model to finish its answer.")

    # free GPU
    del llm, outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ========================================================================
    # 7. PER-QUESTION CORRELATION
    # ========================================================================
    results: List[Dict] = []
    for quid, rows in questions.items():
        if quid not in judged_rank_cnt or judged_rank_cnt[quid] == 0:
            continue   # dropped (too long) or all repeats unparseable
        n = len(rows)
        cnt = judged_rank_cnt[quid]
        # averaged judged rank per original subgraph (lower = better)
        judged_rank_avg = [s / cnt for s in judged_rank_sum[quid]]

        weight_scores = [float(r["weight_score"]) for r in rows]
        weight_rank   = [int(r["weight_rank"])    for r in rows]  # 1 = best

        # Correlate ranks directly. weight_rank: 1=best. judged_rank_avg:
        # low=best. Both oriented "smaller = better", so a POSITIVE rho means
        # the judge agrees with the ensemble weighting.
        rho, tau = rank_correlations(weight_rank, judged_rank_avg)

        gold = (rows[0].get("gold_label") or "").strip().lower()
        cons = (rows[0].get("consensus_answer") or "").strip().lower()
        answer_correct = bool(gold) and bool(cons) and gold == cons
        rawr = bool(answer_correct and (rho == rho)
                    and rho <= RAWR_RHO_THRESHOLD)

        results.append({
            "question_uid": quid,
            "seed_idx": int(rows[0]["seed_idx"]),
            "scheme":   rows[0]["scheme"],
            "dataset":  rows[0]["dataset"],
            "question_id": rows[0]["question_id"],
            "n_subgraphs": n,
            "answer_correct": answer_correct,
            "weight_scores":  json.dumps(weight_scores),
            "weight_rank":    json.dumps(weight_rank),
            "judged_rank_avg": json.dumps([round(x, 4)
                                           for x in judged_rank_avg]),
            "n_order_repeats_used": cnt,
            "spearman_rho": rho,
            "kendall_tau":  tau,
            "right_answer_wrong_reason": rawr,
        })

    # ---- write per-question results CSV -----------------------------------
    RES_FIELDS = ["question_uid", "seed_idx", "scheme", "dataset",
                  "question_id", "n_subgraphs", "answer_correct",
                  "weight_scores", "weight_rank", "judged_rank_avg",
                  "n_order_repeats_used", "spearman_rho", "kendall_tau",
                  "right_answer_wrong_reason"]
    print(f"\n[stage 7b] writing per-question results: {RESULTS_CSV}")
    with open(RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=RES_FIELDS, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for r in results:
            w.writerow(r)

    # ========================================================================
    # 8. AGGREGATE -- per-seed mean+/-std AND pooled, per (scheme, dataset)
    # ========================================================================
    agg: Dict[Tuple[str, str], Dict[int, List[Dict]]] = defaultdict(
        lambda: defaultdict(list))
    for r in results:
        agg[(r["scheme"], r["dataset"])][r["seed_idx"]].append(r)

    print(f"\n{'='*78}")
    print("[stage 7b] EXPLAINABILITY RESULTS -- judge ranking vs weight ranking")
    print(f"{'='*78}")

    summary_rows: List[Dict] = []
    for (scheme, dataset) in sorted(agg.keys()):
        per_seed = agg[(scheme, dataset)]
        seed_rho_means, seed_tau_means = [], []
        all_rho, all_tau = [], []
        n_total = n_correct = n_rawr = 0
        for sidx, rows in per_seed.items():
            srho = [r["spearman_rho"] for r in rows]
            stau = [r["kendall_tau"]  for r in rows]
            m_rho, _ = mean_std(srho)
            m_tau, _ = mean_std(stau)
            seed_rho_means.append(m_rho)
            seed_tau_means.append(m_tau)
            all_rho.extend(srho)
            all_tau.extend(stau)
            n_total   += len(rows)
            n_correct += sum(1 for r in rows if r["answer_correct"])
            n_rawr    += sum(1 for r in rows if r["right_answer_wrong_reason"])

        rho_mean, rho_std = mean_std(seed_rho_means)
        tau_mean, tau_std = mean_std(seed_tau_means)
        pooled_rho, _ = mean_std(all_rho)
        pooled_tau, _ = mean_std(all_tau)
        rawr_frac = (n_rawr / n_correct) if n_correct else float("nan")

        print(f"\n  scheme={scheme:8s} dataset={dataset:6s}  "
              f"({len(per_seed)} seeds, {n_total} questions)")
        print(f"    Spearman rho : {rho_mean:.4f} +/- {rho_std:.4f} "
              f"(per-seed mean)   pooled={pooled_rho:.4f}")
        print(f"    Kendall  tau : {tau_mean:.4f} +/- {tau_std:.4f} "
              f"(per-seed mean)   pooled={pooled_tau:.4f}")
        if n_correct:
            print(f"    right-answer-wrong-reason (rho<={RAWR_RHO_THRESHOLD}): "
                  f"{n_rawr}/{n_correct} ({100*rawr_frac:.1f}%)")
        else:
            print("    right-answer-wrong-reason: n/a")

        summary_rows.append({
            "scheme": scheme, "dataset": dataset,
            "n_seeds": len(per_seed), "n_questions": n_total,
            "spearman_rho_mean": rho_mean, "spearman_rho_std": rho_std,
            "spearman_rho_pooled": pooled_rho,
            "kendall_tau_mean": tau_mean, "kendall_tau_std": tau_std,
            "kendall_tau_pooled": pooled_tau,
            "n_correct": n_correct, "n_right_answer_wrong_reason": n_rawr,
            "right_answer_wrong_reason_frac": rawr_frac,
        })

    print(f"\n{'='*78}")
    print("[stage 7b] FINAL TABLE  (mean +/- std across seeds)")
    print(f"{'='*78}")
    print(f"  {'scheme':10s} {'dataset':8s} {'Spearman rho':>22s} "
          f"{'Kendall tau':>22s} {'RAWR%':>8s}")
    for r in summary_rows:
        rawr = r["right_answer_wrong_reason_frac"]
        rawr_pct = 100 * rawr if rawr == rawr else float("nan")
        print(f"  {r['scheme']:10s} {r['dataset']:8s} "
              f"{r['spearman_rho_mean']:9.3f} +/- {r['spearman_rho_std']:.3f}"
              f"   {r['kendall_tau_mean']:9.3f} +/- {r['kendall_tau_std']:.3f}"
              f"   {rawr_pct:7.1f}%")

    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump({
            "config": {
                "per_subgraph_csv": PER_SUBGRAPH_CSV,
                "judge_model": JUDGE_MODEL,
                "enable_thinking": ENABLE_THINKING,
                "n_order_repeats": N_ORDER_REPEATS,
                "judge_max_tok": JUDGE_MAX_TOK,
                "judge_max_len": JUDGE_MAX_LEN,
                "rawr_rho_threshold": RAWR_RHO_THRESHOLD,
                "questions_dropped_too_long": n_dropped_toolong,
                "judge_outputs_unparseable": n_unparsed,
            },
            "results": summary_rows,
        }, f, indent=2)
    print(f"\n[stage 7b] per-question CSV : {RESULTS_CSV}")
    print(f"[stage 7b] summary JSON     : {SUMMARY_JSON}")
    if n_dropped_toolong:
        print(f"[stage 7b] reminder: {n_dropped_toolong} questions were "
              f"excluded for length -- no chain was ever truncated.")